# Gisborne width-rule comparison and control findings

This notebook reports the committed **real Gisborne District run**. Neither width proxy is MPI's formal centre-line measurement. Erosion is the primary narrow-strip screen, while every proxy disagreement is quarantined for assessor review.

In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

ROOT = Path('..')
candidates = gpd.read_file(ROOT / 'data/processed/gisborne_candidates.gpkg')
comparison = pd.read_csv(ROOT / 'outputs/gisborne/width_method_comparison.csv')
disagreements = comparison[comparison.methods_disagree]
print(f'Input candidates: {len(candidates):,}')
print(f'Width disagreements: {len(disagreements):,} ({len(disagreements)/len(candidates):.2%})')
print('2A/P fail, erosion pass:', len(disagreements[(~disagreements.area_perimeter_pass) & disagreements.erosion_core_pass]))
print('2A/P pass, erosion fail:', len(disagreements[disagreements.area_perimeter_pass & (~disagreements.erosion_core_pass)]))

Input candidates: 5,748
Width disagreements: 694 (12.07%)
2A/P fail, erosion pass: 694
2A/P pass, erosion fail: 0


![Three real Gisborne disagreement cases](../outputs/gisborne/figures/width_disagreement_cases.png)

All 694 disagreements run in the same direction: a local 30 m core survives, but the compactness-sensitive `2A/P` value is below 30 m. The examples show why a surviving core cannot establish *average* width.

In [2]:
import warnings
nztm_pass = candidates.geometry.area >= 10_000
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    naive_wgs84_pass = candidates.to_crs(4326).geometry.area >= 10_000
print(f'NZTM >=1 ha: {nztm_pass.sum():,}')
print(f'Naive WGS84 >=1 ha: {naive_wgs84_pass.sum():,}')
print(f'False rejections: {(nztm_pass & ~naive_wgs84_pass).sum():,}')
print(f'False qualifications: {(~nztm_pass & naive_wgs84_pass).sum():,}')

NZTM >=1 ha: 4,223
Naive WGS84 >=1 ha: 0
False rejections: 4,223
False qualifications: 0


The original brief predicted false qualifications. The real calculation shows the opposite failure mode for this common mistake: all 4,223 true area passes become false rejections because square degrees are tiny compared with 10,000 square metres. The pipeline therefore fails loudly unless the input is EPSG:2193.

In [3]:
labels = pd.read_csv(ROOT / 'outputs/gisborne/review/review_labels.csv')
labels.review_label.value_counts()

review_label
plausible-plantable       13
clearly-not-plantable    12
already-forested           5
Name: count, dtype: int64

Only 13 of 30 sampled candidates were visually plausible at the 2024 imagery date. Five appeared already forested and 12 were constrained by riverbeds, coastal margins, roads, or erosion features. This is a single AI-assisted review without independent ground truth—not a formal accuracy assessment.